In [3]:
import json

rows = []

# 850 positives from service titles
seen = set()
for line in open("data/services.jsonl", encoding="utf-8"):
    s = json.loads(line)
    if s["title"] not in seen:
        seen.add(s["title"])
        rows.append({"text": s["title"], "label": 1})

# your labeled eval questions are positives too
for q in json.load(open("data/eval_questions.json")):
    rows.append({"text": q["query"], "label": 1})

with open("data/topic_train.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"{len(rows)} positives written")

874 positives written


In [4]:
import json

negatives = [
    # greetings
    "salom qalaysiz", "assalomu alaykum yaxshimisiz", "hormang ishlaringiz qalay",
    "салом яхшимисиз", "здравствуйте как дела", "привет что делаешь",
    # weather
    "bugun ob-havo qanday", "ertaga yomg'ir yog'adimi", "havo issiqmi bugun",
    "бугун об-хаво қандай", "какая сегодня погода", "завтра будет дождь",
    # math / nonsense
    "2 qo'shuv 2 nechchi bo'ladi", "asdfghjkl", "100 ni 5 ga bo'l",
    "икки карра икки", "сколько будет пять умножить на шесть", "qwerty123",
    # insults
    "sen ahmoqsan", "bu bot ishlamaydi jinnisan", "foydasiz dastur",
    "сен аҳмоқсан", "ты тупой бот", "дурацкая программа",
    # chitchat
    "eng zo'r restoran qayerda", "kecha futbol kim yutdi", "qanaqa kino ko'ray",
    "энг зўр ресторан қаерда", "какой фильм посмотреть", "кто выиграл вчера в футболе",
    # general knowledge
    "kim prezident", "yer yuzida nechta qit'a bor", "eng baland tog' qaysi",
    "энг катта шаҳар қайси", "какая столица франции", "сколько планет в солнечной системе",
]

rows = json.load(open("data/topic_train.json", encoding="utf-8"))
rows += [{"text": t, "label": 0} for t in negatives]

with open("data/topic_train.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

n_neg = sum(1 for r in rows if r["label"] == 0)
print(f"{len(rows)} total, {n_neg} negatives")

910 total, 36 negatives


In [5]:
import json, random
random.seed(42)

# reload the full positive pool from source
seen, positives = set(), []
for line in open("data/services.jsonl", encoding="utf-8"):
    s = json.loads(line)
    if s["title"] not in seen:
        seen.add(s["title"])
        positives.append(s["title"])
for q in json.load(open("data/eval_questions.json")):
    positives.append(q["query"])

# keep eval questions (high-value), sample titles down to fill ~300
random.shuffle(positives)
positives = positives[:300]

negatives = [r["text"] for r in json.load(open("data/topic_train.json", encoding="utf-8"))
             if r["label"] == 0]

rows = ([{"text": t, "label": 1} for t in positives] +
        [{"text": t, "label": 0} for t in negatives])

with open("data/topic_train.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"{len(rows)} total: {len(positives)} pos, {len(negatives)} neg")

336 total: 300 pos, 36 neg


In [9]:
import sys; sys.path.insert(0, "notebooks")
import rag
import json, numpy as np, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data = json.load(open("data/topic_train.json", encoding="utf-8"))
texts = [d["text"] for d in data]
y = np.array([d["label"] for d in data])

X = rag._embedder.encode(texts, normalize_embeddings=True)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte)))

joblib.dump(clf, "topic_clf.joblib")

/mnt/NewData/vs code/My-gov/mygov/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21240.70it/s]


              precision    recall  f1-score   support

           0       0.88      1.00      0.93         7
           1       1.00      0.98      0.99        61

    accuracy                           0.99        68
   macro avg       0.94      0.99      0.96        68
weighted avg       0.99      0.99      0.99        68



['topic_clf.joblib']

In [ ]:
import joblib
_clf = joblib.load("topic_clf.joblib")

def answer(question, k=5, pool=20):
    emb = rag._embedder.encode(question, normalize_embeddings=True)
    p = _clf.predict_proba([emb])[0][1]        # P(on-topic)
    if p < 0.15:
        return "Menda bu haqda ishonchli ma'lumot yo'q.", []
    # else fall through to retrieval + match-gate as before
    rows = retrieve(question, k, pool)
    if not is_grounded(rows):
        return "Menda bu haqda ishonchli ma'lumot yo'q.", rows
    ...

In [3]:
import sys; sys.path.insert(0, "notebooks")
import rag
import psycopg2
cur = psycopg2.connect(rag.DSN).cursor()
cur.execute("SELECT chunk_index, text FROM chunks WHERE service_id = 667 ORDER BY chunk_index")
for idx, text in cur.fetchall():
    has_price = "foiz" in text.lower() or "BHM" in text or "narx" in text.lower()
    print(f"chunk {idx}: {'💰 PRICE HERE' if has_price else '—'}  ({len(text)} chars)")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 33928.60it/s]


chunk 0: 💰 PRICE HERE  (1382 chars)


In [3]:
import sys; sys.path.insert(0, "notebooks")
import rerank
rows = rerank.retrieve("ikadastr pasprti olish narxi qancha?")
print(f"top rerank score: {rows[0][-1]:.4f}")
print(f"gate threshold:   {rerank.GATE_THRESHOLD}")
print("→ GATE REFUSES" if rows[0][-1] < rerank.GATE_THRESHOLD else "→ passes to LLM")

  [typo-fix] [('ikadastr', 'kadastr'), ('pasprti', 'pasporti')]
top rerank score: 0.2656
gate threshold:   0.3
→ GATE REFUSES


In [4]:
import sys; sys.path.insert(0, "notebooks")
import rerank
import json

qs = json.load(open("data/eval_questions.json", encoding="utf-8"))
print("score   type       query")
for it in sorted(qs, key=lambda q: q["type"]):
    rows = rerank.retrieve(it["query"])
    s = rows[0][-1]
    flag = "⚠️ GRAY" if 0.11 < s < 0.30 else ""
    print(f"{s:+.3f}  {it['type']:10} {it['query'][:40]} {flag}")

score   type       query
+0.025  off_topic  bugun ob-havo qanday? 
+0.002  off_topic  salom, qalaysan? 
  [typo-fix] [('shuv', 'shu')]
+0.001  off_topic  2 qo'shuv 2 nechaga teng? 
+0.001  off_topic  sen ahmoqsan 
+0.001  off_topic  eng zo'r restoran qayerda? 
+0.000  off_topic  kecha futbol o'yini nechida tugadi? 
+0.107  on_topic   ID karta yo'qolsa nima qilaman? 
+0.876  on_topic   tonirovka ruxsatnomasi qanday olinadi? 
  [typo-fix] [('qoraytirishga', 'qoraytirish')]
+0.947  on_topic   avtomobil oynasini qoraytirishga ruxsat  
+0.887  on_topic   yashash joyiga doimiy ro'yxatdan qanday  
+0.994  on_topic   chet elga chiqish pasporti qanday rasmiy 
  [typo-fix] [('safariga', 'safari')]
+0.876  on_topic   haj safariga qanday yozilaman? 
+0.793  on_topic   xorijda tug'ilgan bolani qanday ro'yxatd 
  [typo-fix] [('almashtiraman', 'almashtirish')]
+0.939  on_topic   saylov uchastkasini qanday almashtiraman 
+0.985  on_topic   notarius qabuliga qanday yozilaman? 
+0.879  on_topic   ekspor